In [7]:
import pandas as pd

In [6]:
teams = pd.read_csv("output/teams.csv")
roster = pd.read_csv("output/roster.csv")
roster_counts = roster['TeamId'].value_counts()

team_counts = teams.assign(
    roster_entries=teams['Id'].map(roster_counts).fillna(0).astype(int)
)

below_five = team_counts[team_counts['roster_entries'] < 5]
display(below_five[['Name', 'roster_entries']])



,Name,roster_entries
0,1 Trick Ponies,0
5,34united e-Sports Club,3
6,3sUP Enterprises,4
7,4everzenzyg,2
9,7th heaven X,0
...,...,...
2855,Rising On Gaming,0
2856,CLN Vipers,1
2857,Powned.it,1
2858,Team EasyFix Co-Hop,0


In [4]:
leagues = pd.read_csv("output/leagues.csv")
distinct = leagues['Region'].unique()
print(distinct)

<StringArray>
['International',      'Americas',          'EMEA', 'North America',
          'Asia',        'Europe',        'Brazil',         'Korea',
           'LAN',           'LAS',           'CIS', 'Latin America',
         'China',           'LMS',  'Asia Pacific',           'SEA',
       'Oceania',         'Japan',       'Vietnam',        'Turkey']
Length: 20, dtype: str


In [15]:
import argparse
import csv
import json

import os
os.chdir(r"D:\Works\esport_data")
import importlib
import generate_data
importlib.reload(generate_data)
getStatsForPlayers = generate_data.getStatsForPlayers

def read_names(path):
    with open(path, newline="", encoding="utf-8-sig") as csv_file:
        return [row["Player"].strip() for row in csv.DictReader(csv_file) if row.get("Player", "").strip()]


def main():
    parser = argparse.ArgumentParser(description="Build stats JSON for the players in a list CSV.")
    parser.add_argument("--list", default="players_list.csv")
    parser.add_argument("--out", default="esport-frontend/public/players_list_stats.json")
    args, _ = parser.parse_known_args()

    names = read_names(args.list)
    stats = getStatsForPlayers(names)

    # Keep the CSV order, one entry per name (a name can match several players).
    order = {name.casefold(): index for index, name in enumerate(names)}
    stats.sort(key=lambda player: order.get(player["player"].casefold(), len(order)))

    found = {player["player"].casefold() for player in stats}
    missing = [name for name in names if name.casefold() not in found]

    with open(args.out, "w", encoding="utf-8") as json_file:
        json.dump(stats, json_file, ensure_ascii=False, separators=(",", ":"))
    print(f"{len(stats)}/{len(names)} players written to {args.out}")
    if missing:
        print("No stats found for:", ", ".join(missing))


if __name__ == "__main__":
    main()

50/50 players written to esport-frontend/public/players_list_stats.json
